# Loading HuggingFace Dataset

In [ ]:
import os
import datasets

from datasets import load_dataset
from huggingface_hub import hf_hub_download

ds = load_dataset("LeoChen085/SlipDataset")
# download meta.csv from HuggingFace Hub
meta_path = hf_hub_download(
    repo_id="LeoChen085/SlipDataset",
    filename="meta.csv",
    repo_type="dataset",
)


In [ ]:
from util.dataset import SLIPhfDataset, SLIPCollator
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")

dataset = SLIPhfDataset(
    hf_dataset=ds["train"],
    meta_info=meta_path,  
    patch_size=None,             # None = flexi patch
    text_aug=True,
    is_normalize=True,
    sampling=True,
    subset_ratio=1.0,
)

# use exactly the same collator as before
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=32, collate_fn=SLIPCollator(tokenizer, max_len=128))

In [ ]:
dataset[0].keys()

In [ ]:
example = dataset[0]
print("Caption:", example['caption'])
print("Sensor shape:", example['sensor'].shape)

# Loading Evaluation Dataset

In [ ]:
# ── Zeroshot ──────────────────────────────────────────────────────────────────
from datasets import load_dataset
from torch.utils.data import DataLoader
from util.dataset import ZeroshotDataset
from util.dataset import SLIPCollator

ds = load_dataset("LeoChen085/SlipDataset", name="uci_har")

zs_dataset = ZeroshotDataset(
    data_dir='PPG_CVA', # dataset name
    patch_size=16,
    return_patch=True,
    is_normalize=True,
    hf_repo="LeoChen085/SlipDataset",   # fallback if local data_dir is missing
)

collaten_fn = SLIPCollator(tokenizer, max_len=128)

zs_loader = DataLoader(zs_dataset, batch_size=32, shuffle=False, collate_fn=collaten_fn)


for batch in zs_loader:
    print('zero-shot batch:')
    print(batch.keys())    # ['The subject is walking.', ...]
    break


# ── Linear Probing ────────────────────────────────────────────────────────────
from util.dataset import EvalDataset, EvalCollator
from functools import partial

test_set  = EvalDataset("PPG_CVA", is_train=False, patch_size=16, is_normalize=True,
                        hf_repo="LeoChen085/SlipDataset")

test_loader = DataLoader(
    test_set, batch_size=32, shuffle=False,
    collate_fn=partial(EvalCollator, return_patch=True)
)

for batch in test_loader:
    print('linear probing batch:')
    print(batch.keys())    # ['samples', 'mask', 'time_index', 'targets']
    break

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")
from util.dataset import SftDataset, SFTCollator
# HuggingFace fallback — task_name is inferred from the last component of data_dir
train_set = SftDataset("har_cot", split="val",
                       hf_repo="LeoChen085/SlipSFTDataset")

print('finish loading SFT dataset, now creating dataloader...')
print(len(train_set))  # should print the number of samples in the SFT dataset
data_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=SFTCollator(tokenizer,max_len=128))
for batch in data_loader:
    print('SFT batch:')
    print(batch.keys())    
    break



# Loading Model

In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained("LeoChen085/SLIP", trust_remote_code=True)


In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
state_dict = load_file(hf_hub_download("LeoChen085/SLIP", "har.safetensors"))
model.load_state_dict(state_dict, strict=False)


# Toy Example with Synthetic Tensors

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # Fix macOS OpenMP crash

import torch
from transformers import AutoModel, AutoTokenizer

# ── 1. Load model & tokenizer ──
model = AutoModel.from_pretrained("LeoChen085/SLIP", trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"Model loaded on {device}")

In [ ]:
# ── 2. Build synthetic sensor data ──
# SLIP with patch_size=None (flexi-patch) expects nested lists:
#   sensors["input_ids"]      = [ [tensor(num_p, ps), ...per_var], ...per_batch ]
#   sensors["attention_mask"] = same
#   sensors["time_index"]     = same
# This matches what SLIPCollator produces from the dataloader.

batch_size = 2
num_vars = 3
num_patches = 10
patch_size = 16

sensor_ids, sensor_masks, sensor_times = [], [], []
for _ in range(batch_size):
    vars_x, vars_m, vars_t = [], [], []
    for _ in range(num_vars):
        vars_x.append(torch.randn(num_patches, patch_size, device=device))
        vars_m.append(torch.ones(num_patches, patch_size, device=device))
        vars_t.append(
            torch.linspace(0, 1, num_patches, device=device)
            .unsqueeze(-1).expand(num_patches, patch_size)
        )
    sensor_ids.append(vars_x)
    sensor_masks.append(vars_m)
    sensor_times.append(vars_t)

sensors = {
    "input_ids": sensor_ids,
    "attention_mask": sensor_masks,
    "time_index": sensor_times,
}
print(f"Sensor batch: {batch_size} samples x {num_vars} vars x {num_patches} patches x {patch_size} patch_size")

In [ ]:
# ── 3. Build text batch ──
queries = ["Describe the pattern of this sensor data.", "What activity is this?"]
tok = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length=64)
text = {k: v.to(device) for k, v in tok.items()}
print(f"Text input_ids shape: {text['input_ids'].shape}")

In [ ]:
# ── 4. Contrastive embeddings ──
with torch.no_grad():
    text_emb, sensor_emb = model.get_embedding(text, sensors)

print(f"Text embedding:   {text_emb.shape}")    # (BS, 640)
print(f"Sensor embedding: {sensor_emb.shape}")   # (BS, 640)

# Cosine similarity matrix (text vs sensor)
sim = torch.nn.functional.cosine_similarity(text_emb, sensor_emb)
print(f"Cosine similarity: {sim.tolist()}")

In [ ]:
# ── 5. Generate text conditioned on sensor ──
prompt = "This sensor reading indicates"
gen_tok = tokenizer([prompt] * batch_size, return_tensors="pt", padding=True)
gen_text = {k: v.to(device) for k, v in gen_tok.items()}

with torch.no_grad():
    output_ids = model.generate(gen_text, sensors, max_new_tokens=50)

for i, ids in enumerate(output_ids):
    print(f"Sample {i}: {tokenizer.decode(ids, skip_special_tokens=True)}")

In [ ]:
# ── 6. Sensor-only embedding (no text needed) ──
with torch.no_grad():
    sensor_only_emb = model.get_sensor_embedding(
        input_ids=sensors["input_ids"],
        mask=sensors["attention_mask"],
        time_index=sensors["time_index"],
    )
print(f"Sensor-only embedding: {sensor_only_emb.shape}")  # (BS, 640)

In [ ]:
# ── 7. Swap in a task-specific checkpoint (e.g., HAR) ──
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

har_path = hf_hub_download("LeoChen085/SLIP", "har.safetensors")
result = model.load_state_dict(load_file(har_path, device=str(device)), strict=False)
print(f"Loaded HAR checkpoint — missing: {len(result.missing_keys)}, unexpected: {len(result.unexpected_keys)}")

# Re-run generation with HAR weights
with torch.no_grad():
    output_ids = model.generate(gen_text, sensors, max_new_tokens=50)
print(f"HAR model says: {tokenizer.decode(output_ids[0], skip_special_tokens=True)}")